
# Yambda 13 · Two-stage ranking: iALS + SASRec + popularity → LightGBM

**Stage 1**: top-200 from iALS, top-200 from SASRec, top-100 from last-7-day popularity; union per user.
**Stage 2**: LightGBM LambdaRank on source scores/ranks, recent popularity and trend, artist familiarity,
user activity. Label = candidate is one of the user's organic discoveries.

**Protocol (no leakage)**
* Validation: ranker evaluated with 2-fold cross-validation **by user**. Same time window, disjoint users,
  so a user's own labels never score that user. All ranker decisions are made here.
* Test (`RUN_TEST`, off until we've looked at validation): ranker trained on all validation users,
  applied once to test candidates from models retrained on the test split.

Code: `src/musicrec/ranking.py`. Needs `cands_val_sasrec.parquet` / `cands_test_sasrec.parquet` from
notebook 12 in `reports/yambda/`. ~15 min for validation.


In [3]:

import json, time
import numpy as np
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate, paired_bootstrap, artist_novelty
from musicrec.yambda import load_item_artist
from musicrec.models.baselines import MostPopular
from musicrec.models.als import IALS
from musicrec import ranking as R

pd.set_option("display.float_format", "{:,.4f}".format)
OUT = config.PROJECT_ROOT / "reports" / "yambda"
SOURCES = ["ials", "sasrec", "pop"]
K = {"ials": 200, "sasrec": 200, "pop": 100}
FEATURES = R.feature_columns(SOURCES)
item_artist = load_item_artist()
ials_params = json.loads((OUT / "best_ials.json").read_text())["params"]

def build(split, sasrec_file):
    t = time.time()
    ials = IALS(**ials_params).fit(split)
    pop = MostPopular(by="listeners", recent_days=7).fit(split)
    src = {"ials": R.candidates_from_model(ials, split, K["ials"]),
           "sasrec": pd.read_parquet(OUT / sasrec_file),
           "pop": R.candidates_from_model(pop, split, K["pop"])}
    src["sasrec"] = src["sasrec"][src["sasrec"]["rank"] <= K["sasrec"]]
    c = R.add_labels(R.merge_candidates(src), split)
    c = R.add_features(c, split, item_artist, SOURCES)
    c["rrf"] = R.rrf_scores(c, SOURCES)
    print(f"built {len(c):,} candidate rows in {time.time() - t:.0f}s")
    return c

def score_all(c, split, cols):
    out = {}
    for col, name in cols:
        d = c[["user_idx", "item_idx", col]].dropna()
        out[name] = evaluate(R.PrecomputedScores(d, split.n_items, name, col), split, return_per_user=True)
    return out


## Validation: candidates and their ceiling

In [5]:

val = Split.load(config.YAMBDA_SPLIT_VAL_DIR)
cv = build(val, "cands_val_sasrec.parquet")
R.candidate_recall(cv, val, SOURCES)


built 2,408,617 candidate rows in 251s


ials                    0.1239
sasrec                  0.0931
pop                     0.0445
union                   0.1852
candidates per user   424.2017
dtype: float64


`union` = share of all reachable discoveries that are in anyone's candidate list. That is the ceiling
for re-ranking: a perfect ranker would put every one of them in the top 10 only if it had at most 10.


In [6]:

t = time.time()
cv["ltr"] = R.cross_val_ranker(cv, FEATURES, n_folds=2)
print(f"2-fold LightGBM in {time.time() - t:.0f}s")
res = score_all(cv, val, [("score_ials", "iALS alone"), ("score_sasrec", "SASRec alone"),
                          ("rrf", "RRF blend"), ("ltr", "LightGBM ranker")])
pd.DataFrame({n: {k: v for k, v in r.items() if k != "per_user"} for n, r in res.items()}).T[
    ["recall@10", "ndcg@10", "hit_rate@10", "recall@50", "coverage@10", "avg_pop@10"]]


2-fold LightGBM in 60s


,recall@10,ndcg@10,hit_rate@10,recall@50,coverage@10,avg_pop@10
iALS alone,0.0346,0.0322,0.1671,0.0686,0.0549,832.0774
SASRec alone,0.0268,0.0257,0.1305,0.0494,0.0903,455.4357
RRF blend,0.0387,0.0353,0.1786,0.0740,0.0568,"1,104.2816"
LightGBM ranker,0.0534,0.0506,0.2316,0.0869,0.0683,587.8292


In [7]:

pd.DataFrame({
    "ranker vs iALS": paired_bootstrap(res["iALS alone"]["per_user"], res["LightGBM ranker"]["per_user"]),
    "ranker vs RRF":  paired_bootstrap(res["RRF blend"]["per_user"], res["LightGBM ranker"]["per_user"]),
    "RRF vs iALS":    paired_bootstrap(res["iALS alone"]["per_user"], res["RRF blend"]["per_user"]),
}).T[["a", "b", "diff", "rel_diff", "ci_low", "ci_high", "p(b<=a)"]]


,a,b,diff,rel_diff,ci_low,ci_high,p(b<=a)
ranker vs iALS,0.0346,0.0534,0.0188,0.5445,0.0155,0.0223,0.0000
ranker vs RRF,0.0387,0.0534,0.0147,0.3810,0.0115,0.0180,0.0000
RRF vs iALS,0.0346,0.0387,0.0041,0.1184,0.0016,0.0068,0.0005


In [8]:

# Where does the gain come from? Familiar vs. new artists
new_artist = artist_novelty(val, item_artist)
seg = {}
for col, name in [("score_ials", "iALS alone"), ("ltr", "LightGBM ranker")]:
    d = cv[["user_idx", "item_idx", col]].dropna()
    m = R.PrecomputedScores(d, val.n_items, name, col)
    seg[name] = {lab: evaluate(m, val, targets=val.targets[mask])["recall@10"]
                 for lab, mask in [("known-artist", ~new_artist), ("new-artist", new_artist)]}
pd.DataFrame(seg).T


,known-artist,new-artist
iALS alone,0.0410,0.0160
LightGBM ranker,0.0692,0.0198


In [9]:

# What does the ranker rely on? (model fit on all validation users)
ranker = R.fit_ranker(cv, FEATURES)
imp = pd.Series(ranker.booster_.feature_importance("gain"), index=FEATURES)
(imp / imp.sum()).sort_values(ascending=False).round(3)


item_age_days          0.1220
user_artist_plays      0.1030
rank_ials              0.0800
artist_pop_7d          0.0700
score_ials             0.0550
item_trend             0.0530
score_sasrec           0.0520
user_events_log        0.0510
user_days_since_last   0.0480
user_artist_plays_7d   0.0460
user_events_7d_log     0.0460
score_pop              0.0440
n_sources              0.0430
item_pop_all           0.0410
rank_sasrec            0.0390
item_pop_30d           0.0360
item_pop_7d            0.0360
rank_pop               0.0350
dtype: float64

In [12]:
# 检查 1：B 组用户发现的新歌里，有多少首在 A 组也被发现了（同一时间窗口内的重合）
users = cv["user_idx"].unique()
rng = np.random.default_rng(42)
fold = pd.Series(rng.integers(0, 2, len(users)), index=users).reindex(cv["user_idx"]).to_numpy()
posA = set(cv.loc[(fold == 0) & (cv["label"] == 1), "item_idx"])
posB = cv.loc[(fold == 1) & (cv["label"] == 1), "item_idx"]
print(f"B 组的新发现中，同一首歌也出现在 A 组新发现里的比例: {posB.isin(posA).mean():.1%}")

# 检查 2：去掉所有能识别出具体某首歌的特征，再训练一次排序模型
SAFE = [f for f in FEATURES if not f.startswith(("item_", "artist_pop", "score_pop", "rank_pop"))]
print("保留的特征:", SAFE)
cv["ltr_safe"] = R.cross_val_ranker(cv, SAFE, n_folds=2)
res["LightGBM, no item features"] = score_all(cv, val, [("ltr_safe", "LightGBM, no item features")])["LightGBM, no item features"]
print("recall@10:", round(res["LightGBM, no item features"]["recall@10"], 4))
paired_bootstrap(res["iALS alone"]["per_user"], res["LightGBM, no item features"]["per_user"])

B 组的新发现中，同一首歌也出现在 A 组新发现里的比例: 64.6%
保留的特征: ['score_ials', 'score_sasrec', 'rank_ials', 'rank_sasrec', 'user_artist_plays', 'user_artist_plays_7d', 'user_events_log', 'user_events_7d_log', 'user_days_since_last', 'n_sources']
recall@10: 0.0363


{'a': np.float64(0.03459670295263814),
 'b': np.float64(0.03626500500399761),
 'diff': np.float64(0.0016683020513594661),
 'rel_diff': np.float64(0.04822141733110586),
 'ci_low': np.float64(-0.0009567157701403916),
 'ci_high': np.float64(0.004455217715828843),
 'p(b<=a)': 0.116}


## Test (run once, after we agree on the validation result)

Flip `RUN_TEST = True` and run this cell and the next ones. The ranker is the one fit above on all
validation users; nothing is tuned on test.


In [11]:

RUN_TEST = False
if RUN_TEST:
    test = Split.load(config.YAMBDA_SPLIT_TEST_DIR)
    ct = build(test, "cands_test_sasrec.parquet")
    print(R.candidate_recall(ct, test, SOURCES))
    ct["ltr"] = ranker.predict(ct[FEATURES])
    res_t = score_all(ct, test, [("score_ials", "iALS alone"), ("rrf", "RRF blend"), ("ltr", "LightGBM ranker")])
    final_rank = pd.DataFrame({n: {k: v for k, v in r.items() if k != "per_user"} for n, r in res_t.items()}).T[
        ["recall@10", "ndcg@10", "hit_rate@10", "recall@50", "coverage@10", "avg_pop@10"]]
    sig_t = pd.DataFrame({
        "ranker vs iALS": paired_bootstrap(res_t["iALS alone"]["per_user"], res_t["LightGBM ranker"]["per_user"]),
        "ranker vs RRF": paired_bootstrap(res_t["RRF blend"]["per_user"], res_t["LightGBM ranker"]["per_user"]),
    }).T[["a", "b", "diff", "rel_diff", "ci_low", "ci_high", "p(b<=a)"]]
    final_rank.to_csv(OUT / "final_test_ranker.csv"); sig_t.to_csv(OUT / "final_test_ranker_significance.csv")
    display(final_rank, sig_t)
